In [ ]:
import json
import re
import numpy as np
import os
import sys
from collections import defaultdict
from sklearn.cluster import AgglomerativeClustering
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import StaticEmbedding
from postprocessing import postprocessing
import math
from src.gpt_request import request_api

c:\Users\ASUS\.conda\envs\env_py39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ASUS\.conda\envs\env_py39\lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_lg' (3.4.1) was trained with spaCy v3.4.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [2]:
DISTANCE_THRESHOLD = 0.12

In [3]:
def load_json(path):
    if not os.path.exists(path):
        print(f"Lỗi: Không tìm thấy file tại {path}")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
    

def save_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)
        
        
def extract_references(long_answer):
    if not long_answer:
        return []
    return re.findall(r"\[([^\]]+)\]", long_answer)

def cluster_cohesion(embeddings):
    if len(embeddings) <= 1:
        return 0
    centroid = embeddings.mean(axis=0)
    dists = np.linalg.norm(embeddings - centroid, axis=1)
    return dists.mean()

In [ ]:
def collect_candidates(cluster_file, mapping_data):
    raw_data = load_json(cluster_file)
    processed_data = {}

    for qidx, clusters in raw_data.items():
        cand_list = []
        for cid, cluster in clusters.items():
            if "llm" not in cluster:
                continue

            ans = cluster["llm"].get("short_answer", "").strip().lower()
            long_answer = cluster["llm"].get("long_answer", "")
            context = cluster.get("context", "").lower()
            
            if not ans or ans == "insufficient information":
                continue

            if ans in mapping_data:
                norm_ans = mapping_data[ans]
            else:
                norm_ans = postprocessing(ans)
                norm_ans = mapping_data.get(norm_ans, norm_ans)
                        
            raw_sources = extract_references(long_answer)
            sources = list(set(raw_sources))
   
            cand_list.append({
                "cid": cid,
                "text": norm_ans,
                "long_answer": long_answer,
                "sources": sources,
            })
        
        processed_data[qidx] = cand_list if len(cand_list) >= 1 else None
        
    return processed_data

In [ ]:
import json
import re

def select_best_cluster_via_llm(question_text, raw_clusters, cand_list):
    """
    Sử dụng LLM làm chuyên gia để chọn cụm chứa câu trả lời đúng nhất 
    dựa trên các cặp bằng chứng (Short Answer + Supporting Context).
    """
    if not raw_clusters:
        return None, "No clusters"

    # Bước 1: Xây dựng mô tả chi tiết các cụm (Text-based description)
    clusters_description = ""
    for i, cluster in enumerate(raw_clusters):
        candidate_details = []
        for idx in cluster["indices"]:
            cand = cand_list[idx]
            # Tạo format cặp Short - Long Answer
            detail = f"   + Short Answer: {cand['text']}\n"
            detail += f"     Supporting Context: {cand['long_answer']}" 
            candidate_details.append(detail)
        
        candidates_text = "\n".join(candidate_details)        
        clusters_description += f"--- CLUSTER ID: {i} ---\n"
        clusters_description += f"List of Evidence Pairs in this cluster:\n{candidates_text}\n\n"

    # Bước 2: Chuẩn bị Prompt tiếng Anh chuyên gia
    prompt = f"""You are a Biomedical Expert Evaluator. 
Your task is to identify which cluster contains the **correct answer** to the following medical question.

QUESTION: "{question_text}"

Below are several candidate clusters. Each cluster contains one or more "Evidence Pairs" (a Short Answer and its corresponding Supporting Context). 

INSTRUCTIONS:
1. **Analyze the Question**: Understand the specific biomedical entities and relationships being asked.
2. **Evaluate Evidence Pairs**: For each cluster, verify if the 'Short Answer' is logically and accurately supported by its 'Supporting Context'.
3. **Medical Fact-Checking**: Use your medical knowledge to determine which cluster's information aligns with established scientific facts.
4. **Consistency Check**: A reliable cluster should show a high degree of semantic agreement among its internal evidence pairs.
5. **Selection**: Choose the single cluster ID that provides the most definitive and accurate answer to the question.

CANDIDATE CLUSTERS:
{clusters_description}

Based on your medical expertise and the evidence provided, which cluster is the correct one?

RESPONSE FORMAT (Strict JSON):
{{
  "best_cluster_index": <int>,
  "reason": "<Briefly explain why this cluster's evidence is the most medically sound and accurate for the given question.>"
}}"""

    try:
        # Gọi hàm API thực tế của bạn (request_api cần được định nghĩa trước)
        response_str = request_api(prompt, model='o3-mini') 
        
        # Parse JSON an toàn bằng Regex
        match = re.search(r'\{.*\}', response_str, re.DOTALL)
        if match:
            res_data = json.loads(match.group())
            idx = res_data.get("best_cluster_index")
            reason = res_data.get("reason", "No reason provided")
            
            if 0 <= idx < len(raw_clusters):
                selected_cluster = raw_clusters[idx]
                selected_cluster["llm_reason"] = reason
                selected_cluster["cluster_index"] = idx # Lưu lại index để debug
                return selected_cluster, "llm_decision"
                
    except Exception as e:
        print(f"Lỗi khi LLM chọn cụm: {e}")
        
    # Bước 3: Fallback nếu LLM gặp lỗi (Chọn cụm có tổng số nguồn lớn nhất)
    fallback = max(raw_clusters, key=lambda x: x["unique_source"])
    fallback["llm_reason"] = "Fallback to Max Sources (API Error)"
    return fallback, "fallback_max_source"


In [ ]:
def cluster_and_evaluate(metadata, processed_data, model, dict_golden):
    fused_results = {}
    list_id_miss = set()
    detailed_logs = {} # Lưu log chi tiết để kiểm tra "hit" golden

    for qidx, cand_list in processed_data.items():
        # Bỏ qua nếu không có candidate hoặc chỉ có 1 (không cần phân cụm/chọn)
        if not cand_list or len(cand_list) < 2:
            fused_results[qidx] = {"abstain": True}
            list_id_miss.add(qidx)
            continue
            
        question_text = metadata[qidx]['original question']
        # Lấy danh sách đáp án đúng (Golden) của câu hỏi này
        golden_answers = [str(g).lower().strip() for g in dict_golden.get(qidx, {}).get("True", [])]
        
        # --- BƯỚC 1: PHÂN CỤM (Giữ nguyên logic của bạn) ---
        texts = [c["text"] for c in cand_list]
        embeddings = model.encode(texts)
        raw_clusters = []

        labels = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=0.12, 
            metric="cosine", linkage="average"
        ).fit(embeddings).labels_

        for lbl in set(labels):
            indices = [i for i, l in enumerate(labels) if l == lbl]
            total_refs = sum([len(cand_list[i]["sources"]) for i in indices])
            raw_clusters.append({
                "indices": indices,
                "evidence_count": len(indices),
                "unique_source": total_refs
            })
        
        # --- BƯỚC 2: GỬI TẤT CẢ CANDIDATE VÀO LLM ĐỂ CHỌN CỤM ---
        best_cluster, method = select_best_cluster_via_llm(question_text, raw_clusters, cand_list)

        # --- BƯỚC 3: KIỂM TRA XEM CỤM ĐƯỢC CHỌN CÓ CHỨA ĐÁP ÁN ĐÚNG KHÔNG ---
        if best_cluster:
            chosen_indices = best_cluster["indices"]
            
            # Kiểm tra: Chỉ cần 1 candidate trong cụm khớp với golden thì coi là đúng
            cluster_contains_golden = False
            matched_candidate = None
            
            for idx in chosen_indices:
                cand_text = str(cand_list[idx]["text"]).lower().strip()
                if cand_text in golden_answers:
                    cluster_contains_golden = True
                    matched_candidate = cand_list[idx]["text"]
                    break # Tìm thấy 1 cái là đủ

            fused_results[qidx] = {
                "final_answer": None, # Tạm để None theo yêu cầu
                "abstain": False,
                "is_correct_cluster": cluster_contains_golden, # Flag đánh giá chất lượng chọn cụm
                "debug_info": {
                    "method": method,
                    "reason": best_cluster.get("llm_reason"),
                    "hit_golden": cluster_contains_golden
                }
            }

            # Cache lại chi tiết để bạn soi lại xem LLM chọn đúng hay sai
            detailed_logs[qidx] = {
                "question": question_text,
                "selected_cluster_idx": best_cluster.get("cluster_index"),
                "llm_reason": best_cluster.get("llm_reason"),
                "is_correct_cluster": cluster_contains_golden,
                "matched_with_golden_text": matched_candidate,
                "golden_answers": golden_answers,
                "all_candidates_in_selected_cluster": [cand_list[i]["text"] for i in chosen_indices]
            }
        else:
            fused_results[qidx] = {"abstain": True}
            list_id_miss.add(qidx)

    # Lưu log chi tiết
    os.makedirs('cache_logs', exist_ok=True)
    save_json(detailed_logs, 'cache_logs/full_cluster_llm_logs.json')

    return fused_results, list(list_id_miss)

In [ ]:
def compute_selected_summary(fused_results, dict_golden):
    total_with_evidence = len(fused_results)
    selected, correct_selected, total_wrong = 0, 0, 0
    list_id_wrong = []

    for qidx, info in fused_results.items():
        if info.get("abstain", True):
            continue

        selected += 1
        
        # Thay vì check pred in golden_list, ta check flag is_correct_cluster
        if info.get("is_correct_cluster", False):
            correct_selected += 1
        else:
            list_id_wrong.append(qidx)
            total_wrong += 1

    acc = correct_selected / selected if selected else 0.0
    coverage = selected / total_with_evidence if total_with_evidence else 0.0

    return {
        "total": total_with_evidence, "selected": selected, "correct": correct_selected,
        "acc": acc, "coverage": coverage, "wrong": total_wrong
    }, list_id_wrong

In [7]:
def load_pubmed_model():
    print("Loading embedding model (PubMedBERT)...")    
    return SentenceTransformer("pritamdeka/S-BioBert-snli-multinli-stsb",device="cpu")

In [ ]:
BASE_FOLDER = "src/cluster/final"
PATH_GOLDEN = "data/all_answer_golden_56.json"
PATH_MAPPING = 'data/mapping_answer_gol.json'

PATH_CLUSTER = r'src\cluster\final\WH_answer_kmean_merge_google_wiki_7_only_GG.json' #only gg

In [ ]:
model = load_pubmed_model()
mapping_data = load_json(PATH_MAPPING)
dict_golden = load_json(PATH_GOLDEN)

metadata  = load_json(r'data\metadata_medhop.json')

Loading embedding model (PubMedBERT)...


In [ ]:
print("Step 1: Collecting candidates...")
processed_candidates_mapping = collect_candidates(PATH_CLUSTER, mapping_data)

Step 1: Collecting candidates...


In [ ]:
results, list_id_miss = cluster_and_evaluate(processed_candidates_mapping, model, dict_golden)


In [ ]:
#vtmthu MAPPING
print(len(list_id_miss))
save_json(list_id_miss, f"list_id_miss_ct.json")
stats, list_id_wrong = compute_selected_summary(results, dict_golden)
print(f"Selected Answers   : {stats['selected']} (Coverage: {stats['coverage']*100:.2f}%)")
print(f"Correct Answers    : {stats['correct']} (Accuracy: {stats['acc']*100:.2f}%)")


MIN_SCORE_THRESHOLD:  1.0
42
Selected Answers   : 430 (Coverage: 91.10%)
Correct Answers    : 395 (Accuracy: 91.86%)
